In [3]:
%pip install -q "transformers==4.57.1"
%pip install -q --no-deps "bitsandbytes==0.48.2"

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import gc
import hashlib
import json
import re
import unicodedata
from decimal import Decimal
from pathlib import Path
from time import perf_counter

import bitsandbytes as bnb
import pandas as pd
import torch
from PIL import Image
from peft import PeftModel, prepare_model_for_kbit_training
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    GenerationConfig,
    Qwen3VLForConditionalGeneration
)

In [6]:
def read_json(path):
    return path.read_text(encoding="utf-8") and json.loads(
        path.read_text(encoding="utf-8")
    )

def save_json(path, data):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    tmp.replace(path)

def sha256(path):
    with path.open("rb") as file:
        return hashlib.file_digest(file, "sha256").hexdigest()

In [7]:
ROOT = Path("/content/drive/MyDrive/MLLM/mordern_llm")
DATA = ROOT / "data/sroie"
BEST = ROOT / "outputs/qlora_run1/best_adapter"
OUT = ROOT / "outputs/evaluation/day5"
OUT.mkdir(parents=True, exist_ok=True)

recipe = read_json(BEST / "training_recipe.json")
base_cfg = recipe["base"]
task_config = read_json(BEST / "task_config.json")
manifest_path = DATA / "manifest.json"
manifest = read_json(manifest_path)

DOC_PROMPT = task_config["prompt"]
IMAGE_KWARGS = task_config["image_kwargs"]
FIELDS = tuple(task_config["fields"])

test_records = manifest["test"]
test_ids = [r["id"] for r in test_records]

saved_example = read_json(BEST / "validation_rows.json")[0]
warmup_record = next(r for r in manifest["val"] if r["id"] == saved_example["id"])

DEVICE = torch.device("cuda:0")
processor = AutoProcessor.from_pretrained(BEST, use_fast=recipe["processor_use_fast"])

run_config = {
    "base": base_cfg,
    "task": task_config,
    "manifest_sha256": sha256(manifest_path),
    "adapter_sha256": sha256(BEST / "adapter_model.safetensors"),
    "gpu": torch.cuda.get_device_name(DEVICE),
    "torch_version": str(torch.__version__)
}

print("Fixed test receipts:", len(test_ids))
print("Selected epoch:", read_json(BEST / "best_metrics.json")["epoch"])

Fixed test receipts: 100
Selected epoch: 1.0


In [8]:
def load_base(for_training):
    quant_config = BitsAndBytesConfig.from_dict(
        base_cfg["quantization_config"]
    )
    net = Qwen3VLForConditionalGeneration.from_pretrained(
        base_cfg["model_id"],
        revision=base_cfg["revision"],
        quantization_config=quant_config,
        dtype=torch.float16,
        device_map={"": 0},
        attn_implementation="sdpa",
        low_cpu_mem_usage=True
    )
    
    net = prepare_model_for_kbit_training(
        net,
        use_gradient_checkpointing=for_training,
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    
    net.model.visual.to(dtype=torch.float16)
    net.config.use_cache = not for_training
    net.config.text_config.use_cache = not for_training
    return net


def normal(field, value):
    if not isinstance(value, str) or not value.strip():
        return None

    text = " ".join(
        unicodedata.normalize("NFKC", value).casefold().split()
    )
    if field != "total":
        return text
    
    text = re.sub(r"^(?:rm|\$)\s*", "", text)
    if not re.fullmatch(
        r"[+-]?(?:\d+|\d{1,3}(?:,\d{3})+)(?:\.\d+)?",
        text
    ):
        return None

    amount = Decimal(text.replace(",", ""))
    return format(amount.normalize(), "f")

In [9]:
def score_doc(raw, reference):
    try:
        json.loads(raw)
        raw_valid = True
    except ValueError:
        raw_valid = False
    
    text = raw.strip()
    lines = text.splitlines()
    wrapped = (
        len(lines) >= 3
        and lines[0].strip().lower() in ("```json", "```")
        and lines[-1].strip() == "```"
    )
    
    if wrapped:
        text = "\n".join(lines[1:-1])
    
    try:
        prediction = json.loads(text)
    except ValueError:
        prediction = None

    schema = (
        isinstance(prediction, dict)
        and set(prediction) == set(FIELDS)
        and all(
            isinstance(value, str) or value is None
            for value in prediction.values()
        )
    )
    correct = {}
    for field in FIELDS:
        if not schema:
            correct[field] = False
        elif prediction[field] is None or reference[field] is None:
            correct[field] = prediction[field] is reference[field]
        else:
            pred = normal(field, prediction[field])
            gold = normal(field, reference[field])
            correct[field] = pred is not None and pred == gold

    return {
        "prediction": prediction,
        "raw_json_valid": raw_valid,
        "schema_valid": schema,
        "used_fence_cleanup": wrapped,
        **{field + "_correct": value for field, value in correct.items()},
        "all_correct": all(correct.values())
    }

In [10]:
def read_rgb(path):
    with Image.open(path) as source:
        return source.convert("RGB")

def receipt_image(record):
    path = Path(record["image_path"])
    if not path.is_file():
        path = DATA / "images" / path.name
    return read_rgb(path)

generation = GenerationConfig(
    max_new_tokens=task_config["max_new_tokens"],
    do_sample=False,
    use_cache=True,
    eos_token_id=processor.tokenizer.eos_token_id,
    pad_token_id=processor.tokenizer.pad_token_id,
    bos_token_id=processor.tokenizer.bos_token_id
)

@torch.no_grad()
def answer_image(net, image, prompt=DOC_PROMPT):
    messages = [{
        "role" : "user",
        "content" : [
            {"type": "image"},
            {"type": "text", "text": prompt}
        ]
    }]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    batch = processor(
        text=[text],
        images=[image],
        images_kwargs=IMAGE_KWARGS,
        add_special_tokens=False,
        return_tensors="pt"
    ).to(DEVICE)
    
    with torch.autocast("cuda", dtype=torch.float16):
        output = net.generate(
            **batch,
            generation_config=generation,
            use_model_defaults=False
        )
    
    new_ids = output[:, batch["input_ids"].shape[1]:]
    return processor.batch_decode(
        new_ids, skip_special_tokens=True
    )[0].strip()

In [ ]:
IMAGE_DIR = Path("/content/drive/MyDrive/MLLM/data/images")
def find_image(prefix):
    matches = [
        p for p in IMAGE_DIR.glob(f"{prefix}*")
        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
    ]

    return str(matches[0])

flower = find_image("0c1be4")
city = find_image("c7fa0a")
food = find_image("335eb4")

VQA_CASES = [
    {
        "id": "flower_color",
        "image_path": flower,
        "question": "What color is the flower? Answer in one word.",
        "reference": "yellow"
    },
    {
        "id": "flower_subject",
        "image_path": flower,
        "question": "Which is the main subject, a flower or a car? Answer flower or car.",
        "reference": "flower"
    },
    {
        "id": "city_viewpoint",
        "image_path": city,
        "question": "Is the city viewed from above or from street level? Answer above or street level.",
        "reference": "above"
    },
    {
        "id": "city_street_level",
        "image_path": city,
        "question": "Was this photo taken at street level? Answer only yes or no.",
        "reference": "no"
    },
    {
        "id": "food_subject",
        "image_path": food,
        "question": "Which is the main subject, food or a vehicle? Answer food or vehicle.",
        "reference": "food"
    },
]

print("VQA questions:", len(VQA_CASES))
print("Distinct Drive images:", len({r["image_path"] for r in VQA_CASES}))

VQA questions: 5
Distinct Drive images: 3


**Test evaluation function**

In [ ]:
def evaluate_receipts(net, setting):
    path = OUT / f"{setting}_receipts.json"
    state = (
        read_json(path) if path.exists()
        else {"config": run_config, "rows": []}
    )
    
    rows = state["rows"]
    done = len(rows)
    
    for record in test_records[done:]:
        image = receipt_image(record)
        
        torch.cuda.synchronize(DEVICE)
        torch.cuda.reset_peak_memory_stats(DEVICE)
        start = perf_counter()
        
        answer = answer_image(net, image)
        torch.cuda.synchronize(DEVICE)
        latency = perf_counter() - start
        peak = torch.cuda.max_memory_allocated(DEVICE) / 2**30
        
        rows.append({
            "id": record["id"],
            "reference": record["reference"],
            "answer": answer,
            **score_doc(answer, record["reference"]),
            "latency_s": latency,
            "peak_gpu_GiB": peak
        })
        
        if len(rows) % 10 == 0 or len(rows) == len(test_records):
            save_json(path, state)
            print(f"{setting}: {len(rows)}/{len(test_records)} saved")

    return rows


def evaluate_vqa(net):
    return [
        {
            **case,
            "answer": answer_image(net, read_rgb(case["image_path"]), case["question"])
        }
        for case in VQA_CASES
    ]

**Base → adapter → adapted**

In [14]:
warmup_image = receipt_image(warmup_record)
net = load_base(for_training=False).eval()

_ = answer_image(net, warmup_image)
base_rows = evaluate_receipts(net, "base_nf4")
base_vqa = evaluate_vqa(net)

net = PeftModel.from_pretrained(
    net, BEST, is_trainable=False
).eval()

reloaded_answer = answer_image(net, warmup_image)

print("Active adapter:", net.active_adapter)
print("Reloaded validation answer:", reloaded_answer)
print("Same as saved validation answer:", reloaded_answer == saved_example["answer"],)

adapted_rows = evaluate_receipts(net, "adapted_nf4")
adapted_vqa = evaluate_vqa(net)

del net
gc.collect()
torch.cuda.empty_cache()

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


base_nf4: 10/100 saved
base_nf4: 20/100 saved
base_nf4: 30/100 saved
base_nf4: 40/100 saved
base_nf4: 50/100 saved
base_nf4: 60/100 saved
base_nf4: 70/100 saved
base_nf4: 80/100 saved
base_nf4: 90/100 saved
base_nf4: 100/100 saved
Active adapter: default
Reloaded validation answer: {"company": "ANZEL ADVERTISING (M) SDN BHD", "date": "27/03/2018", "total": "8.50"}
Same as saved validation answer: True
adapted_nf4: 10/100 saved
adapted_nf4: 20/100 saved
adapted_nf4: 30/100 saved
adapted_nf4: 40/100 saved
adapted_nf4: 50/100 saved
adapted_nf4: 60/100 saved
adapted_nf4: 70/100 saved
adapted_nf4: 80/100 saved
adapted_nf4: 90/100 saved
adapted_nf4: 100/100 saved


**Metrics and paired comparison**

In [ ]:
metric_keys = [
    "raw_json_valid", "schema_valid",
    "company_correct", "date_correct", "total_correct",
    "all_correct"
]

def summarize(setting, rows):
    df = pd.DataFrame(rows)
    return {
        "setting": setting,
        "receipts": len(df),
        **{f"{key}_pct": 100 * df[key].mean() for key in metric_keys},
        "mean_latency_s": df["latency_s"].mean(),
        "peak_gpu_GiB": df["peak_gpu_GiB"].max()
    }

summary = pd.DataFrame([summarize("base_nf4", base_rows), summarize("adapted_nf4", adapted_rows),])
base_df = pd.DataFrame(base_rows)[["id", "reference", "answer", "all_correct"]]
adapted_df = pd.DataFrame(adapted_rows)[["id", "answer", "all_correct"]]

paired = base_df.merge(
    adapted_df, on="id",
    suffixes=("_base", "_adapted"),
    validate="one_to_one"
)

def outcome(base_ok, adapted_ok):
    if base_ok:
        return "both_correct" if adapted_ok else "lost"
    return "gained" if adapted_ok else "both_wrong"

paired["outcome"] = [outcome(a, b) for a, b in zip(paired["all_correct_base"], paired["all_correct_adapted"])]
counts = paired["outcome"].value_counts().reindex(
    ["both_correct", "gained", "lost", "both_wrong"], fill_value=0
)

display(summary.round(3))
display(counts.to_frame("receipts"))

delta_pp = 100 * (counts["gained"] - counts["lost"]) / len(paired)
print(f"All-fields-correct change: {delta_pp:+.1f} percentage points")

changes = paired[paired["outcome"].isin(["gained", "lost"])]
display(changes.groupby("outcome", sort=False).head(2))

summary.to_csv(OUT / "summary.csv", index=False)
paired.to_csv(OUT / "paired_receipts.csv", index=False)

if VQA_CASES:
    vqa = pd.DataFrame(base_vqa).merge(
        pd.DataFrame(adapted_vqa)[["id", "answer"]],
        on="id",
        suffixes=("_base", "_adapted"),
        validate="one_to_one"
    )
    display(vqa[["question", "reference", "answer_base", "answer_adapted"]])
    vqa.to_csv(OUT / "vqa_regression.csv", index=False)
else:
    print("Section B VQA regression: not run")

,setting,receipts,raw_json_valid_pct,schema_valid_pct,company_correct_pct,date_correct_pct,total_correct_pct,all_correct_pct,mean_latency_s,peak_gpu_GiB
0,base_nf4,100,0.0,100.0,81.0,88.0,91.0,65.0,5.363,3.445
1,adapted_nf4,100,100.0,100.0,86.0,95.0,98.0,79.0,6.876,3.476


,receipts
outcome,
both_correct,64
gained,15
lost,1
both_wrong,20


All-fields-correct change: +14.0 percentage points


,id,reference,answer_base,all_correct_base,answer_adapted,all_correct_adapted,outcome
3,X51005433518,"{'company': 'UNIHAKKA INTERNATIONAL SDN BHD', ...","```json\n{\n ""company"": ""UNIHAKKA INTERNATION...",False,"{""company"": ""UNIHAKKA INTERNATIONAL SDN BHD"", ...",True,gained
44,X51006389894,"{'company': 'KING'S CONFECTIONERY S/B', 'date'...","```json\n{\n ""company"": ""KING'S CONFECTIONERY...",False,"{""company"": ""KING'S CONFECTIONERY S/B"", ""date""...",True,gained
62,X51006619328,"{'company': 'YIN MA (M) SDN.BHD.', 'date': '31...","```json\n{\n ""company"": ""YIN MA (M) SDN.BHD.""...",True,"{""company"": ""YIN MA (M) SDN. BHD."", ""date"": ""3...",False,lost


,question,reference,answer_base,answer_adapted
0,What color is the flower? Answer in one word.,yellow,Yellow,Yellow
1,"Which is the main subject, a flower or a car? ...",flower,flower,flower
2,Is the city viewed from above or from street l...,above,above,above
3,Was this photo taken at street level? Answer o...,no,no,no
4,"Which is the main subject, food or a vehicle? ...",food,food,food


In [16]:
images = {case["id"]: case["image_path"] for case in VQA_CASES}
flower = read_rgb(images["flower_color"])
food = read_rgb(images["food_subject"])

object_cases = [
    ("flower_present", flower, "Is there a flower?", "yes"),
    ("dog_absent", flower, "Is there a dog?", "no"),
    ("food_present", food, "Is there food?", "yes"),
    ("car_absent", food, "Is there a car?", "no")
]

document_cases = [
    ("receipt_visible", receipt_image(warmup_record), warmup_record["reference"]),
    ("no_receipt", flower, dict.fromkeys(FIELDS))
]

In [17]:
def run_probes(net, setting):
    rows = []
    for name, image, question, expected in object_cases:
        raw = answer_image(net, image, question + "Answer only yes or no.")
        answer = raw.strip().casefold().rstrip(".!?")
        
        if answer == expected:
            outcome = "correct"
        elif answer == "yes":
            outcome = "unsupported_yes"
        elif answer == "no":
            outcome = "missed_present"
        else:
            outcome = "non_yes_no"

        rows.append([setting, name, expected, raw, outcome])
        
    
    for name, image, expected in document_cases:
        raw = answer_image(net, image, DOC_PROMPT)
        score = score_doc(raw, expected)

        if not score["schema_valid"]:
            outcome = "invalid_output"
        elif score["all_correct"]:
            outcome = "correct_abstention" if name == "no_receipt" else "correct"
        elif name == "no_receipt":
            outcome = "unsupported_field"
        elif any(score["prediction"][field] is None for field in FIELDS):
            outcome = "unnecessary_abstention"
        else:
            outcome = "field_error"

        rows.append([setting, name, expected, raw, outcome])

    return rows

**Receipt extraction API**

In [18]:
%pip install -q fastapi python-multipart httpx

In [19]:
from contextlib import asynccontextmanager
from io import BytesIO
from threading import Lock
from fastapi import FastAPI, File, Form, HTTPException, Request, UploadFile
from fastapi.testclient import TestClient

MAX_BYTES = 8 * 1024 * 1024
MAX_PIXELS = 12_000_000
gpu_lock = Lock()

@asynccontextmanager
async def lifespan(app):
    app.state.net = PeftModel.from_pretrained(
        load_base(for_training=False), BEST, is_trainable=False
    ).eval()
    try:
        yield
    finally:
        del app.state.net

In [20]:
app = FastAPI(lifespan=lifespan)

@app.post("/extract")
def extract(
    request: Request,
    image: UploadFile = File(),
    instruction: str = Form(DOC_PROMPT, max_length=1000)
):
    data = image.file.read(MAX_BYTES + 1)
    if len(data) > MAX_BYTES:
        raise HTTPException(413, "Image file too large")
    
    try:
        with Image.open(BytesIO(data)) as source:
            if source.width * source.height > MAX_PIXELS:
                raise HTTPException(413, "Image has too many pixels")
            rgb = source.convert("RGB")
        
    except (OSError, Image.DecompressionBombError):
        raise HTTPException(400, "Invalid image")
    
    with gpu_lock:
        raw = answer_image(request.app.state.net, rgb, instruction)
    
    try:
        fields = json.loads(raw)
    except json.JSONDecodeError:
        raise HTTPException(502, "Model did not return raw JSON")

    if not (isinstance(fields, dict) and set(fields) == set(FIELDS) and all(v is None or isinstance(v, str) for v in fields.values())):
        raise HTTPException(502, "Model output does not match receipt schema")

    return {"fields": fields}

In [21]:
path = Path(warmup_record["image_path"])
if not path.is_file():
    path = DATA / "images" / path.name

with TestClient(app) as client:
    with app.state.net.disable_adapter():
        rows = run_probes(app.state.net, "base_nf4")

    rows += run_probes(app.state.net, "adapted_nf4")
    display(pd.DataFrame(rows, columns=["setting", "probe", "expected", "answer", "outcome"]))

    with path.open("rb") as file:
        start = perf_counter()
        response = client.post("/extract", files={"image": (path.name, file)})
        elapsed = perf_counter() - start

    print("Receipt:", response.status_code, f"{elapsed:.2f}s", response.json())

    if response.status_code == 200:
        score = score_doc(
            json.dumps(response.json()["fields"]),
            warmup_record["reference"]
        )
        print("All fields correct:", score["all_correct"])

    bad = client.post("/extract", files={"image": ("bad.jpg", b"not an image")})
    print("Invalid upload:", bad.status_code, bad.json())

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


,setting,probe,expected,answer,outcome
0,base_nf4,flower_present,yes,yes,correct
1,base_nf4,dog_absent,no,no,correct
2,base_nf4,food_present,yes,yes,correct
3,base_nf4,car_absent,no,no,correct
4,base_nf4,receipt_visible,"{'company': 'ANZEL ADVERTISING (M) SDN BHD', '...","```json\n{\n ""company"": ""Anzel Advertising (M...",field_error
5,base_nf4,no_receipt,"{'company': None, 'date': None, 'total': None}","{""company"": ""ABC Retail"", ""date"": ""2023-04-15""...",unsupported_field
6,adapted_nf4,flower_present,yes,yes,correct
7,adapted_nf4,dog_absent,no,no,correct
8,adapted_nf4,food_present,yes,yes,correct
9,adapted_nf4,car_absent,no,no,correct


/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Receipt: 200 6.58s {'fields': {'company': 'ANZEL ADVERTISING (M) SDN BHD', 'date': '27/03/2018', 'total': '8.50'}}
All fields correct: True
Invalid upload: 400 {'detail': 'Invalid image'}
